# Lab Day 2 — DeepWeeds: backbone, công thức huấn luyện, suy luận (bản làm lại theo đúng khung `starter/`)

Notebook này chạy toàn bộ Bước 0 → 5 bằng **code trong `code/`** (đã hoàn thiện từ `starter/`), dùng **một hàm `train.run(Config)`** cho mọi thí nghiệm, và **`eval.py` gốc của repo** để tính chỉ số.

**Quy tắc:** mọi lựa chọn trên **val**; **test chỉ chạy ở Bước 4, một lần mỗi seed**.

Mọi kết quả (`runs/`, `curves/`, `predictions/`, `results.xlsx`) ghi thẳng lên Google Drive → Colab bị ngắt thì **chạy lại từ đầu**: các lần chạy đã xong sẽ được bỏ qua (`skip_if_done=True`).

Thứ tự: chạy lần lượt từ trên xuống. Các ô huấn luyện dài có ghi thời gian ước tính.

## 0. Môi trường, code và dữ liệu

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, dataclasses, platform, subprocess
WORK = "/content/drive/MyDrive/DeepWeeds_Lab2"          # nơi lưu bền (Drive)
CODE_DIR = f"{WORK}/code"                               # code/ đã hoàn thiện (upload sẵn lên Drive)
REPO_DIR = "/content/K4-Track4-Day2-Deeplearning-Advance"  # repo bài lab (lấy eval.py GỐC, không sửa)
for d in ("runs", "curves", "predictions", "figures", "eval_out"):
    os.makedirs(f"{WORK}/{d}", exist_ok=True)

!pip -q install timm openpyxl
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/VinUni-AI20k/K4-Track4-Day2-Deeplearning-Advance.git {REPO_DIR}

sys.path.insert(0, REPO_DIR)     # eval.py gốc
sys.path.insert(0, CODE_DIR)     # code/ của bạn
os.chdir(WORK)                   # đường dẫn tương đối runs/ curves/ predictions/ nằm trên Drive

import numpy as np, pandas as pd, torch, timm
import eval as ev
print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU -> Runtime > Change runtime type > GPU")
print("eval.py:", ev.__file__)

In [ ]:
# Tải dữ liệu: images.zip từ Zenodo (kiểm tra MD5) + 4 file nhãn fold 0 từ GitHub của tác giả.
# Bản zip được cache trên Drive để lần sau không phải tải lại.
import hashlib, shutil, glob
DATA = "/content/data"
os.makedirs(f"{DATA}/labels", exist_ok=True)
ZIP_DRIVE = f"{WORK}/images.zip"
MD5 = "b7b30f96d466fba86016aa5a26606e0f"

def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

URLS = ["https://zenodo.org/api/records/7939060/files/images.zip/content",
        "https://zenodo.org/records/7939060/files/images.zip?download=1"]
if not (os.path.exists(ZIP_DRIVE) and md5(ZIP_DRIVE) == MD5):
    for url in URLS:
        !curl -L --retry 5 -A "Mozilla/5.0" -o /content/images.zip "{url}"
        got = md5("/content/images.zip")
        if got == MD5:
            break
        print("Tải lỗi từ", url, "| kích thước", os.path.getsize("/content/images.zip"), "byte | nội dung đầu:")
        print(open("/content/images.zip", "rb").read(400))
    if got != MD5:
        # Zenodo chặn IP của Colab (403 Forbidden) -> dùng bản images.zip do chính tác giả DeepWeeds đăng trên Google Drive
        # (link trong README của github.com/AlexOlsen/DeepWeeds). MD5 vẫn được kiểm tra với giá trị của Zenodo.
        !curl -L --retry 3 -o /content/images.zip "https://drive.usercontent.google.com/download?id=1xnK3B6K6KekDI55vwJ0vnc2IGoDga9cj&export=download&confirm=t"
        got = md5("/content/images.zip")
        print(f"[GHI VÀO BÁO CÁO] Zenodo trả 403 cho Colab; dùng images.zip của tác giả trên Google Drive: MD5 {got}")
    assert got == MD5, f"MD5 sai: {got}"
    shutil.copy("/content/images.zip", ZIP_DRIVE)
print("MD5 OK:", MD5)

if len(glob.glob(f"{DATA}/**/*.jpg", recursive=True)) < 17509:
    !unzip -q -n {ZIP_DRIVE} -d {DATA}/images_raw
# tìm thư mục chứa nhiều .jpg nhất
jpgs = glob.glob(f"{DATA}/**/*.jpg", recursive=True)
IMAGES_DIR = pd.Series([os.path.dirname(p) for p in jpgs]).value_counts().index[0]
print("IMAGES_DIR =", IMAGES_DIR, "| số ảnh:", len(os.listdir(IMAGES_DIR)))

LABELS_DIR = f"{DATA}/labels"
BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    if not os.path.exists(f"{LABELS_DIR}/{name}.csv"):
        !wget -q -O {LABELS_DIR}/{name}.csv {BASE}/{name}.csv
!ls -la {LABELS_DIR}

In [ ]:
# Kiểm tra tự viết (focal γ=0 ≡ CE, CutMix trộn ảnh+nhãn đúng diện tích, gộp BN, param_groups, EMA...)
# và bộ test của repo (eval.py + khung starter). Không cần GPU.
!cd {CODE_DIR} && python -m unittest tests_code 2>&1 | tail -3
!cd {REPO_DIR} && python -m unittest discover -s tests 2>&1 | tail -3

## Bước 0 — Kiểm tra split, EDA, kiểm tra pipeline

In [ ]:
from dataset import load_split, check_split, CLASS_NAMES, build_transforms, make_loader, denormalize
import matplotlib.pyplot as plt
from PIL import Image

train_df, val_df, test_df = load_split(LABELS_DIR, fold=0)
split_info = check_split(train_df, val_df, test_df, IMAGES_DIR)   # assert nếu giao khác rỗng / thiếu file / hợp != 17.509

# Phân bố lớp (toàn bộ 17.509 ảnh) + đối chiếu Table 1 bài báo
full = pd.concat([train_df, val_df, test_df])
cnt = full.Label.value_counts().sort_index()
paper = [1125, 1064, 1031, 1022, 1062, 1009, 1074, 1016, 9106]   # Table 1, Olsen et al. 2019
eda = pd.DataFrame({"class": CLASS_NAMES, "count": cnt.values, "paper_table1": paper,
                    "train": split_info["per_class"]["train"], "val": split_info["per_class"]["val"],
                    "test": split_info["per_class"]["test"]})
eda["share_%"] = (eda["count"] / eda["count"].sum() * 100).round(2)
display(eda)
print(f"Tỉ lệ lớp nhiều nhất / ít nhất = {cnt.max()/cnt.min():.2f}  |  khớp Table 1: {list(cnt.values)==paper}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(CLASS_NAMES, cnt.values, color=["tab:green"]*8 + ["tab:gray"])
for i, v in enumerate(cnt.values): ax.text(i, v + 80, str(v), ha="center", fontsize=8)
ax.set(title="DeepWeeds: số ảnh mỗi lớp (17.509 ảnh)", ylabel="số ảnh"); plt.xticks(rotation=30, ha="right")
plt.tight_layout(); plt.savefig("figures/eda_class_distribution.png", dpi=130); plt.show()

# 3 ảnh mỗi lớp
fig, axs = plt.subplots(9, 3, figsize=(7, 20))
for c in range(9):
    for j, fn in enumerate(train_df[train_df.Label == c].Filename.sample(3, random_state=0)):
        axs[c, j].imshow(Image.open(f"{IMAGES_DIR}/{fn}")); axs[c, j].axis("off")
    axs[c, 0].set_title(CLASS_NAMES[c], loc="left", fontsize=9)
plt.tight_layout(); plt.savefig("figures/eda_samples.png", dpi=110); plt.show()

sizes = pd.Series([Image.open(f"{IMAGES_DIR}/{f}").size + (Image.open(f"{IMAGES_DIR}/{f}").mode,)
                   for f in train_df.Filename.sample(300, random_state=0)]).value_counts()
print("Kích thước/mode ảnh (300 ảnh mẫu):", sizes.to_dict())

In [ ]:
# Kiểm tra pipeline trước khi chạy thật: seed, loss ban đầu ≈ ln 9, overfit 1 batch nhỏ,
# ảnh sau augmentation khớp nhãn, train()/eval() đúng lúc.
import math, torch.nn.functional as F
from train import set_seed
from model import build_model, param_groups, set_train_mode
dev = torch.device("cuda")
set_seed(0)
m = build_model("resnet50", pretrained=True, num_classes=9).to(dev)
dl = make_loader(train_df, IMAGES_DIR, build_transforms(True, 224), 32, train=True, num_workers=2)
x, y, f = next(iter(dl)); x, y = x.to(dev), y.to(dev)
m.eval()
with torch.no_grad(): l0 = F.cross_entropy(m(x), y).item()
print(f"[1] loss ban đầu = {l0:.3f}  (kỳ vọng ≈ ln 9 = {math.log(9):.3f})")

opt = torch.optim.AdamW(param_groups(m, 1e-4, 1e-3, 0.0))
xs, ys = x[:16], y[:16]
for it in range(80):
    set_train_mode(m); opt.zero_grad(); loss = F.cross_entropy(m(xs), ys); loss.backward(); opt.step()
print(f"[2] overfit 16 ảnh sau 80 bước: loss = {loss.item():.4f} (phải gần 0)")
m.eval()
with torch.no_grad(): print("    accuracy trên chính 16 ảnh ở eval():", (m(xs).argmax(1) == ys).float().mean().item())
print("[3] model.training sau .eval():", m.training)
del m, opt; torch.cuda.empty_cache()

fig, axs = plt.subplots(2, 6, figsize=(14, 5))
for a, img, lab in zip(axs.ravel(), denormalize(x[:12].cpu()), y[:12].cpu()):
    a.imshow(img.permute(1, 2, 0)); a.set_title(CLASS_NAMES[lab], fontsize=8); a.axis("off")
plt.suptitle("[4] Ảnh sau augmentation (đã giải chuẩn hoá) + nhãn"); plt.tight_layout()
plt.savefig("figures/check_augmentation.png", dpi=110); plt.show()

from losses import mix_batch
xm, (ya, yb, lam) = mix_batch(x[:6].cpu(), y[:6].cpu(), 1.0, "cutmix")
fig, axs = plt.subplots(1, 6, figsize=(14, 3))
for i, a in enumerate(axs):
    a.imshow(denormalize(xm[i]).permute(1, 2, 0)); a.axis("off")
    a.set_title(f"{CLASS_NAMES[ya[i]][:10]} / {CLASS_NAMES[yb[i]][:10]}\nlam={lam:.2f}", fontsize=7)
plt.suptitle("CutMix: ảnh và nhãn cùng được trộn"); plt.tight_layout()
plt.savefig("figures/check_cutmix.png", dpi=110); plt.show()

## Bước 1 — So sánh backbone (công thức nền T00, seed 0)
Đủ ràng buộc: ResNet (resnet50), ResNeXt + ConvNeXt, transformer (Swin-T, DeiT-S), mạng nhẹ (EfficientNet-B0, MobileNetV3-L).
Mỗi lần chạy ghi `runs/B0x/seed0/` và `curves/B0x_<tên>.png`. Thời gian/epoch được đo và in ra → dùng để lập kế hoạch GPU.
Thiếu GPU: bỏ bớt dòng `B06` (DeiT) hoặc đặt `EPOCHS = 10` và ghi vào báo cáo.

In [ ]:
from train import Config, run
import lab_steps as LS
EPOCHS = 12
COMMON = dict(images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, epochs=EPOCHS, num_workers=2)

BACKBONES = [
    ("B01", "resnet50"),
    ("B02", "resnext50_32x4d"),
    ("B03", "convnext_tiny"),
    ("B04", "swin_tiny_patch4_window7_224"),
    ("B05", "efficientnet_b0"),
    ("B06", "mobilenetv3_large_100"),
    ("B07", "deit_small_patch16_224"),
]
for exp_id, name in BACKBONES:
    run(Config(exp_id=exp_id, backbone=name, seed=0, **COMMON))

S = LS.load_summaries("runs")
btab = LS.backbone_table(S)
display(btab.drop(columns=["curve"]))

In [ ]:
# Biểu đồ đánh đổi macro-F1 val ↔ độ trễ / số tham số
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for _, r in btab.iterrows():
    ax[0].scatter(r.latency_b1_p50_ms, r.val_macro_f1); ax[0].annotate(r.backbone.split("_patch")[0], (r.latency_b1_p50_ms, r.val_macro_f1), fontsize=8)
    ax[1].scatter(r.params_M, r.val_macro_f1); ax[1].annotate(r.backbone.split("_patch")[0], (r.params_M, r.val_macro_f1), fontsize=8)
ax[0].set(xlabel="độ trễ batch 1 FP32, p50 (ms)", ylabel="macro-F1 val", title="F1 theo độ trễ")
ax[1].set(xlabel="số tham số (M)", ylabel="macro-F1 val", title="F1 theo kích thước")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.savefig("figures/backbone_tradeoff.png", dpi=130); plt.show()

# Chọn backbone đi tiếp CHỈ dựa trên val (1 seed -> chỉ coi là "tốt hơn" khi chênh lệch rõ)
best = btab.sort_values("val_macro_f1", ascending=False).iloc[0]
light = btab[btab.backbone.isin(["efficientnet_b0", "mobilenetv3_large_100"])].sort_values("val_macro_f1").iloc[-1]
BEST_ID, BEST_BACKBONE = best.exp_id, best.backbone
print(f"Backbone tốt nhất theo macro-F1 val: {BEST_BACKBONE} ({BEST_ID}) = {best.val_macro_f1:.4f}, p50 {best.latency_b1_p50_ms:.1f} ms")
print(f"Mạng nhẹ tốt nhất: {light.backbone} = {light.val_macro_f1:.4f}, p50 {light.latency_b1_p50_ms:.1f} ms")
# Nếu muốn ưu tiên cân bằng F1/độ trễ thì đổi tay BEST_ID/BEST_BACKBONE ở đây và GHI LÝ DO vào báo cáo.

## Bước 2 — Công thức huấn luyện trên backbone đã chọn
`T00` = công thức nền trên `BEST_BACKBONE`. Seed 0 của T00 **chính là** lần chạy B0x (cùng cấu hình, cùng seed) nên được chép lại, không train lại (ghi rõ trong báo cáo).
`T00` seed 1, 2 dùng để **đo nhiễu do seed** (std) — đồng thời là mốc cho Bước 4.
Mỗi `T0x` chỉ khác `T00` **một** yếu tố. Thứ tự ưu tiên: ô dưới chạy nhóm bắt buộc trước; nhóm tuỳ chọn để sau nếu còn GPU.

In [ ]:
LS.clone_run("runs", BEST_ID, "T00", seed=0, curves_dir="curves")
T_BASE = dict(backbone=BEST_BACKBONE, **COMMON)

# Nhiễu do seed của công thức nền
for s in (1, 2):
    run(Config(exp_id="T00", seed=s, **T_BASE))

AXES = {}
def T(exp_id, axis, **change):
    AXES[exp_id] = axis
    return run(Config(exp_id=exp_id, seed=0, desc=f"{BEST_BACKBONE.split('_patch')[0]}_" + "_".join(f"{k}-{v}" for k, v in change.items()),
                      **{**T_BASE, **change}))

# --- bắt buộc: >= 3 trục, có loss và augmentation ---
T("T01", "A. khởi tạo", init="frozen")
T("T02", "B. augmentation", aug="trivial")
T("T03", "B. augmentation", mix="cutmix")
T("T04", "C. loss", loss="ls", label_smoothing=0.1)
T("T05", "C. loss", loss="focal", focal_gamma=2.0)
T("T06", "C. loss", loss="ce_weighted")
T("T07", "F. chính quy hoá", ema_decay=0.999)

In [ ]:
# --- tuỳ chọn (chạy nếu còn GPU) ---
T("T08", "A. khởi tạo", init="scratch")
T("T09", "D. cân bằng mẫu", sampler="balanced")
T("T10", "E. LR", lr_head=1e-4)              # cùng LR cho head và backbone
T("T11", "B. augmentation", aug="color")

In [ ]:
# Bảng ablation + so Δ với nhiễu (std của T00 qua 3 seed)
S = LS.load_summaries("runs")
mean0, std0, n0 = LS.noise_std(S, "T00")
print(f"T00: macro-F1 val = {mean0:.4f} ± {std0:.4f} ({n0} seed)  -> ngưỡng 'khác biệt rõ' = 2·std = {2*std0:.4f}")
ttab = LS.training_table(S, AXES)
display(ttab.drop(columns=["curve"]))

# Kết hợp (tham lam theo trục): lấy MỖI TRỤC giá trị tốt nhất nếu nó hơn T00 rõ (Δ > 2·std)
winners = {}
for _, r in ttab[(ttab.seed == 0) & (ttab.exp_id != "T00")].iterrows():
    if r.delta_vs_T00 > 2 * std0 and (r.axis not in winners or r.delta_vs_T00 > winners[r.axis][1]):
        winners[r.axis] = (r.exp_id, r.delta_vs_T00)
print("Yếu tố thắng rõ theo trục:", winners)
combo = {}
for ax_, (eid, _) in winners.items():
    c = LS.cfg_of("runs", eid); base = Config(**T_BASE)
    combo.update({k: v for k, v in dataclasses.asdict(c).items()
                  if k not in ("exp_id", "seed", "desc") and getattr(base, k) != v})
if not combo:   # không yếu tố nào vượt nhiễu: vẫn thử kết hợp 2 yếu tố có Δ dương lớn nhất (để trả lời câu hỏi cộng dồn)
    for eid in ttab[(ttab.seed == 0) & (ttab.exp_id != "T00")].sort_values("delta_vs_T00").exp_id.tail(2):
        c = LS.cfg_of("runs", eid); base = Config(**T_BASE)
        combo.update({k: v for k, v in dataclasses.asdict(c).items() if k not in ("exp_id", "seed", "desc") and getattr(base, k) != v})
print("Cấu hình kết hợp T12:", combo)
AXES["T12"] = "kết hợp"
run(Config(exp_id="T12", seed=0, desc=f"{BEST_BACKBONE.split('_patch')[0]}_combo", **{**T_BASE, **combo}))

In [ ]:
S = LS.load_summaries("runs")
ttab = LS.training_table(S, AXES)
display(ttab.drop(columns=["curve"]))
# Công thức tốt nhất mang sang Bước 3/4: macro-F1 val cao nhất (seed 0). Nếu không vượt T00 quá 2·std, ghi rõ trong báo cáo.
cand = ttab[ttab.seed == 0].sort_values("val_macro_f1", ascending=False)
FINAL_RECIPE_ID = cand.iloc[0].exp_id
FINAL_CFG = LS.cfg_of("runs", FINAL_RECIPE_ID)
print("Công thức tốt nhất (val):", FINAL_RECIPE_ID, "|", cand.iloc[0].diff_vs_T00, "| macro-F1 val", round(cand.iloc[0].val_macro_f1, 4))

## Bước 3 — Suy luận trên **val** (không train lại) và đo độ trễ
I00 1 view · I01 TTA lật (gộp xác suất) · I03 TTA lật (gộp logit) · I02 5-crop / 10-crop · I04 dò độ phân giải · I05 ensemble 2 backbone · I06 EMA · I07 temperature scaling + ECE · I08 gộp BN / FP16.

In [ ]:
from benchmark import latency_report, tta_latency
from train import load_trained
import inference as inf
dev = torch.device("cuda")
model = load_trained(FINAL_CFG, dev)
rows, VAL = [], {}
lat1 = latency_report(model, 1, 224, "fp32", "cuda", warmup=10, iters=100)

for meth, (_, _, _, space, K) in LS.METHODS.items():
    names, y, ll, space = LS.run_method(model, val_df, IMAGES_DIR, meth, dev)
    VAL[meth] = (y, ll, space)
    m_ = LS.metrics(ev, y, LS.calibrated_probs(ll, 1.0, space))
    lat = lat1 if K == 1 else latency_report(model, 1, 224, "fp32", "cuda", warmup=10, iters=100, views=K)
    rows.append({"exp_id": meth.split("_")[0], "method": meth, "model": FINAL_RECIPE_ID, "K": K,
                 "val_macro_f1": m_["macro_f1"], "val_top1": m_["top1"], "val_ece": m_["ece"],
                 "p50_ms": lat["p50"], "p95_ms": lat["p95"], "p99_ms": lat["p99"],
                 "images_per_s_b1": lat["images_per_s"], "rel_cost_vs_I00": lat["p50"] / lat1["p50"]})

# I04: dò độ phân giải (CNN nhận mọi cỡ; ViT/Swin cố định cỡ -> ghi 'không áp dụng')
for r in (224, 256, 288, 320):
    try:
        names, y, ll, space = LS.run_resolution(model, val_df, IMAGES_DIR, r, dev)
        VAL[f"I04_res{r}"] = (y, ll, space)
        m_ = LS.metrics(ev, y, LS.calibrated_probs(ll))
        lat = latency_report(model, 1, r, "fp32", "cuda", warmup=10, iters=100)
        rows.append({"exp_id": "I04", "method": f"I04_res{r}", "model": FINAL_RECIPE_ID, "K": 1,
                     "val_macro_f1": m_["macro_f1"], "val_top1": m_["top1"], "val_ece": m_["ece"],
                     "p50_ms": lat["p50"], "p95_ms": lat["p95"], "p99_ms": lat["p99"],
                     "images_per_s_b1": lat["images_per_s"], "rel_cost_vs_I00": lat["p50"] / lat1["p50"]})
    except Exception as e:
        print(f"I04 res{r}: không áp dụng cho {FINAL_CFG.backbone} ({type(e).__name__}: {str(e)[:80]})")

INF = pd.DataFrame(rows); display(INF)

In [ ]:
# I05 ensemble: mô hình chung kết + backbone khác tốt nhì ở Bước 1 (dùng logit val đã lưu, không cần GPU cho F1)
y0 = np.load(f"runs/{FINAL_RECIPE_ID}/seed0/val_labels.npy")
p_final = inf.softmax_np(np.load(f"runs/{FINAL_RECIPE_ID}/seed0/val_logits.npy"))
second = btab[btab.backbone != FINAL_CFG.backbone].sort_values("val_macro_f1").iloc[-1]
y2 = np.load(f"runs/{second.exp_id}/seed0/val_labels.npy"); assert (y0 == y2).all()
p_ens = inf.ensemble_probs([p_final, inf.softmax_np(np.load(f"runs/{second.exp_id}/seed0/val_logits.npy"))])
m_ = LS.metrics(ev, y0, p_ens)
m2 = load_trained(LS.cfg_of("runs", second.exp_id), dev)
lat2 = latency_report(m2, 1, 224, "fp32", "cuda", warmup=10, iters=100); del m2
rows.append({"exp_id": "I05", "method": f"I05_ensemble[{FINAL_RECIPE_ID}+{second.exp_id}:{second.backbone}]",
             "model": f"{FINAL_RECIPE_ID}+{second.exp_id}", "K": 2, "val_macro_f1": m_["macro_f1"], "val_top1": m_["top1"],
             "val_ece": m_["ece"], "p50_ms": lat1["p50"] + lat2["p50"], "p95_ms": lat1["p95"] + lat2["p95"],
             "p99_ms": lat1["p99"] + lat2["p99"], "images_per_s_b1": 1000 / (lat1["p50"] + lat2["p50"]),
             "rel_cost_vs_I00": (lat1["p50"] + lat2["p50"]) / lat1["p50"]})

# I06 EMA: T07 (có EMA) so với trọng số thường của cùng lần chạy -> không tốn thêm chi phí suy luận
h = pd.read_csv("runs/T07/seed0/history.csv") if os.path.exists("runs/T07/seed0/history.csv") else None
if h is not None:
    b = h.loc[h.val_macro_f1.idxmax()]
    rows.append({"exp_id": "I06", "method": "I06_EMA_weights (T07)", "model": "T07", "K": 1,
                 "val_macro_f1": b.val_macro_f1, "val_top1": b.val_top1, "val_ece": b.val_ece,
                 "p50_ms": lat1["p50"], "p95_ms": lat1["p95"], "p99_ms": lat1["p99"],
                 "images_per_s_b1": lat1["images_per_s"], "rel_cost_vs_I00": 1.0,
                 "note": f"cùng epoch, trọng số thường: macro-F1 {b.val_macro_f1_raw:.4f}"})

# I07 temperature scaling: T khớp trên VAL; ECE trước/sau (cùng val) và ECE 'trung thực' (khớp nửa này, đo nửa kia)
y, ll, space = VAL["I00_1view"]
T_val = LS.fit_T(ll, y, space)
e_before = ev.ece_score(LS.calibrated_probs(ll, 1.0), y); e_after = ev.ece_score(LS.calibrated_probs(ll, T_val), y)
cf_b, cf_a = LS.crossfit_ece(ev, ll, y)
m_ = LS.metrics(ev, y, LS.calibrated_probs(ll, T_val))
rows.append({"exp_id": "I07", "method": f"I07_temperature T={T_val:.3f}", "model": FINAL_RECIPE_ID, "K": 1,
             "val_macro_f1": m_["macro_f1"], "val_top1": m_["top1"], "val_ece": e_after,
             "p50_ms": lat1["p50"], "p95_ms": lat1["p95"], "p99_ms": lat1["p99"], "images_per_s_b1": lat1["images_per_s"],
             "rel_cost_vs_I00": 1.0, "note": f"ECE val trước {e_before:.4f} -> sau {e_after:.4f}; khớp chéo 2 nửa: {cf_b:.4f} -> {cf_a:.4f}"})
print(f"T = {T_val:.3f} | ECE val {e_before:.4f} -> {e_after:.4f} | khớp chéo {cf_b:.4f} -> {cf_a:.4f}")

# Biểu đồ độ tin cậy trước/sau
def reliability(p, y, ax, title):
    conf, pred = p.max(1), p.argmax(1); bins = np.linspace(0, 1, 16); mids, accs = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        k = (conf > lo) & (conf <= hi)
        if k.sum(): mids.append(conf[k].mean()); accs.append((pred[k] == y[k]).mean())
    ax.plot([0, 1], [0, 1], "k--", lw=.8); ax.plot(mids, accs, "o-"); ax.set(title=title, xlabel="độ tin cậy", ylabel="accuracy"); ax.grid(alpha=.3)
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
reliability(LS.calibrated_probs(ll, 1.0), y, ax[0], f"Trước (ECE {e_before:.4f})")
reliability(LS.calibrated_probs(ll, T_val), y, ax[1], f"Sau T={T_val:.2f} (ECE {e_after:.4f})")
plt.tight_layout(); plt.savefig("figures/calibration_val.png", dpi=130); plt.show()

In [ ]:
# I08: gộp BN và FP16/AMP — độ chính xác và độ trễ. Đo trên mô hình chung kết + một CNN có BN (ResNet-50 B01, mạng nhẹ).
LAT, x_chk = [], torch.randn(2, 3, 224, 224, device=dev)
def lat_rows(tag, mdl, fused):
    for dt in ("fp32", "amp", "fp16"):
        for bs in (1, 32):
            r = latency_report(mdl, bs, 224, dt, "cuda", warmup=10, iters=100 if bs == 1 else 50, fused_bn=fused)
            LAT.append({"config": tag, **{k: r[k] for k in ("gpu", "dtype", "batch", "img_size", "fused_bn", "p50", "p95", "p99", "images_per_s", "torch", "preprocessing")}})

cnn_ids = [FINAL_RECIPE_ID] + [i for i, n in BACKBONES if n in ("resnet50", light.backbone)]
for eid in dict.fromkeys(cnn_ids):
    mdl = load_trained(LS.cfg_of("runs", eid), dev).float()
    fused = inf.fuse_conv_bn(mdl, check_input=x_chk)
    lat_rows(f"{eid} ({mdl.default_cfg['architecture']})", mdl, False)
    if fused.n_fused:
        lat_rows(f"{eid} ({mdl.default_cfg['architecture']}) + BN fused", fused, True)
        names, y, lg = inf.predict_logits(fused, LS._loader(val_df, IMAGES_DIR, "std", 224), dev)
        m_ = LS.metrics(ev, y, inf.softmax_np(lg))
        print(f"{eid}: gộp BN -> macro-F1 val {m_['macro_f1']:.4f}, sai số đầu ra {fused.fuse_max_abs_diff:.1e}")
    else:
        print(f"{eid}: không có cặp conv+BN (LayerNorm) -> gộp BN không áp dụng")
    if eid == FINAL_RECIPE_ID:   # FP16 có đổi độ chính xác / hiệu chuẩn không?
        names, y, lg = inf.predict_logits(mdl.half(), LS._loader(val_df, IMAGES_DIR, "std", 224), dev, view=lambda t: t.half())
        m_ = LS.metrics(ev, y, inf.softmax_np(lg)); mdl.float()
        fp16 = [l for l in LAT if l["config"].startswith(eid) and l["dtype"] == "fp16" and l["batch"] == 1 and not l["fused_bn"]][0]
        rows.append({"exp_id": "I08", "method": "I08_fp16", "model": eid, "K": 1, "val_macro_f1": m_["macro_f1"],
                     "val_top1": m_["top1"], "val_ece": m_["ece"], "p50_ms": fp16["p50"], "p95_ms": fp16["p95"],
                     "p99_ms": fp16["p99"], "images_per_s_b1": fp16["images_per_s"], "rel_cost_vs_I00": fp16["p50"] / lat1["p50"]})
    del mdl, fused; torch.cuda.empty_cache()

LATENCY = pd.DataFrame(LAT); display(LATENCY)
INF = pd.DataFrame(rows); display(INF)
fig, ax = plt.subplots(figsize=(8, 5))
for _, r in INF.iterrows():
    ax.scatter(r.p95_ms, r.val_macro_f1); ax.annotate(r.method[:22], (r.p95_ms, r.val_macro_f1), fontsize=7)
ax.axvline(100, color="r", ls=":", label="ngân sách 100 ms"); ax.set(xlabel="p95 batch 1 (ms)", ylabel="macro-F1 val", title="Đánh đổi độ chính xác – độ trễ (val)")
ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.savefig("figures/inference_tradeoff.png", dpi=130); plt.show()

In [ ]:
# Chốt phương pháp suy luận cho chung kết CHỈ trên val: macro-F1 val cao nhất trong các phương pháp
# áp dụng được cho MỘT mô hình (I00–I04); temperature scaling (T khớp trên val) luôn được áp dụng sau đó.
single = INF[INF.method.isin(list(LS.METHODS) + [m for m in INF.method if m.startswith("I04_res")])]
FINAL_METHOD = single.sort_values("val_macro_f1", ascending=False).iloc[0].method
RT = single[single.p95_ms <= 100].sort_values("val_macro_f1", ascending=False).iloc[0]
print("Phương pháp suy luận chung kết:", FINAL_METHOD)
print(f"Cấu hình thời gian thực (p95 ≤ 100 ms): {RT.method}, p95 = {RT.p95_ms:.1f} ms")

## Bước 4 — Chung kết: ≥ 3 seed, TEST một lần mỗi seed
- `F01` = công thức `FINAL_RECIPE_ID` + suy luận `FINAL_METHOD` + temperature scaling (T khớp trên val của từng seed).
- Mốc = `T00` + `I00` (1 view, không hiệu chuẩn), cùng 3 seed. T00 đã train ở Bước 1–2 → chỉ nạp checkpoint và chạy test một lần.
- **Không quay lại sửa gì sau khi chạy ô này.**

In [ ]:
FINAL_ROWS = []
for seed in (0, 1, 2):
    # mốc: T00 + I00 -> predictions/T00_seed{k}_test.csv
    run(Config(exp_id="T00", seed=seed, **T_BASE, save_test_predictions=True))
    # chung kết: train công thức tốt nhất với seed k (F01raw_* = 1 view, không T), rồi áp suy luận đã chốt
    fcfg = Config(**{**dataclasses.asdict(FINAL_CFG), "exp_id": "F01", "seed": seed, "pred_tag": "F01raw",
                     "desc": f"{FINAL_CFG.backbone.split('_patch')[0]}_final", "save_test_predictions": True})
    run(fcfg)
    if not os.path.exists(f"predictions/F01_seed{seed}_test.csv"):
        FINAL_ROWS.append(LS.final_predictions(ev, fcfg, FINAL_METHOD, IMAGES_DIR, LABELS_DIR, "predictions", dev, tag="F01"))
    else:
        FINAL_ROWS.append(json.load(open(f"runs/F01/seed{seed}/F01_final.json")))
display(pd.DataFrame(FINAL_ROWS))

In [ ]:
import json
TC, LC = f"{LABELS_DIR}/test_subset0.csv", f"{LABELS_DIR}/labels.csv"
for tag in ("F01", "F01uncal", "F01raw", "T00"):
    !python {REPO_DIR}/eval.py score --pred "predictions/{tag}_seed*_test.csv" --test-csv {TC} --labels {LC} --tag {tag} --out eval_out
RT_P95 = float(RT.p95_ms)
!python {REPO_DIR}/eval.py grade --final "predictions/F01_seed*_test.csv" --baseline "predictions/T00_seed*_test.csv" \
    --uncal "predictions/F01uncal_seed*_test.csv" --final-val "predictions/F01_seed*_val.csv" \
    --latency-p95-ms {RT_P95} --test-csv {TC} --val-csv {LABELS_DIR}/val_subset0.csv --labels {LC} --out eval_out

In [ ]:
# Phân tích lỗi: ma trận nhầm lẫn (tổng 3 seed), cặp nhầm nhiều nhất, ảnh bị đoán sai (Chinee apple ↔ Snake weed)
cm = pd.read_csv("eval_out/F01_confusion_sum.csv", index_col=0).values
cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(8, 7)); im = ax.imshow(cmn, cmap="Blues")
ax.set_xticks(range(9), CLASS_NAMES, rotation=45, ha="right"); ax.set_yticks(range(9), CLASS_NAMES)
for i in range(9):
    for j in range(9): ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7, color="white" if cmn[i, j] > .5 else "black")
ax.set(xlabel="dự đoán", ylabel="nhãn thật", title="F01 test — ma trận nhầm lẫn (cộng 3 seed)"); plt.colorbar(im)
plt.tight_layout(); plt.savefig("figures/confusion_F01_test.png", dpi=130); plt.show()
off = cm.copy(); np.fill_diagonal(off, 0); i, j = np.unravel_index(off.argmax(), off.shape)
print(f"Cặp nhầm nhiều nhất: {CLASS_NAMES[i]} -> {CLASS_NAMES[j]} ({off[i, j]} ảnh / 3 seed)")
PERCLASS = pd.read_csv("eval_out/F01_per_class.csv"); display(PERCLASS)

p0 = pd.read_csv("predictions/F01_seed0_test.csv")
wrong = p0[(p0.y_true != p0.y_pred) & (p0.y_true.isin([0, 7]))].head(12)
fig, axs = plt.subplots(2, 6, figsize=(15, 6))
for a, (_, r) in zip(axs.ravel(), wrong.iterrows()):
    a.imshow(Image.open(f"{IMAGES_DIR}/{r.Filename}")); a.axis("off")
    a.set_title(f"thật: {CLASS_NAMES[r.y_true]}\nđoán: {CLASS_NAMES[r.y_pred]} ({r[f'p{r.y_pred}']:.2f})", fontsize=7)
plt.suptitle("Ảnh test bị đoán sai của Chinee apple / Snake weed (seed 0)"); plt.tight_layout()
plt.savefig("figures/errors_hard_classes.png", dpi=110); plt.show()

## Bước 5 — `results.xlsx`

In [ ]:
S = LS.load_summaries("runs")
btab = LS.backbone_table(S)
ttab = LS.training_table(S, AXES)

def eo(tag): return json.load(open(f"eval_out/{tag}_summary.json"))
def per_seed(tag): return pd.read_csv(f"eval_out/{tag}_per_seed.csv")
final = []
for tag, desc in (("F01", f"{FINAL_CFG.backbone} + {FINAL_RECIPE_ID} ({LS.diff_from_base(dataclasses.asdict(FINAL_CFG), dataclasses.asdict(Config(**T_BASE)))}) + {FINAL_METHOD} + temperature"),
                  ("T00", f"{BEST_BACKBONE} + T00 + I00 (mốc)")):
    ps = per_seed(tag)
    for _, r in ps.iterrows():
        vrow = (pd.read_csv(f"predictions/F01_seed{int(r.seed)}_val.csv") if tag == "F01" else None)
        vf1 = (ev.compute_metrics(vrow.y_true.values, vrow.y_pred.values, vrow[[f"p{i}" for i in range(9)]].values)["macro_f1"]
               if tag == "F01" else LS.summary_of("runs", "T00", int(r.seed))["val_macro_f1"])
        final.append({"exp_id": tag, "config": desc, "seed": int(r.seed), "val_macro_f1": vf1,
                      "test_macro_f1": r.macro_f1, "test_top1": r.top1, "test_ece": r.ece, "test_balanced_acc": r.balanced_acc})
    s = eo(tag)
    final.append({"exp_id": tag, "config": "MEAN ± STD", "seed": "mean±std",
                  "test_macro_f1": f"{s['macro_f1']['mean']:.4f} ± {s['macro_f1']['std']:.4f}",
                  "test_top1": f"{s['top1']['mean']:.4f} ± {s['top1']['std']:.4f}",
                  "test_ece": f"{s['ece']['mean']:.4f} ± {s['ece']['std']:.4f}"})
FINAL = pd.DataFrame(final)

pc = []
for tag in ("F01", "T00"):
    d = pd.read_csv(f"eval_out/{tag}_per_class.csv")
    for _, r in d.iterrows():
        pc.append({"config": tag, "class": r["class"], "n_test": r.support, "precision": r.precision_mean,
                   "recall": r.recall_mean, "f1": r.f1_mean, "f1_std": r.f1_std})
PER = pd.DataFrame(pc)

allrows = pd.concat([
    btab.assign(kind="backbone", cost=btab.latency_b1_p50_ms)[["exp_id", "kind", "backbone", "val_macro_f1", "val_top1", "cost", "params_M"]],
    ttab[ttab.seed == 0].assign(kind="training", cost=np.nan)[["exp_id", "kind", "backbone", "val_macro_f1", "val_top1", "diff_vs_T00"]],
    INF.assign(kind="inference", backbone=FINAL_CFG.backbone, cost=INF.p50_ms)[["exp_id", "kind", "method", "val_macro_f1", "val_top1", "cost", "p95_ms"]],
])
SUMMARY = allrows.sort_values("val_macro_f1", ascending=False).head(10).rename(columns={"cost": "latency_b1_p50_ms"})
f, t = eo("F01"), eo("T00")
SUMMARY = pd.concat([SUMMARY, pd.DataFrame([
    {"exp_id": "F01 (test, 3 seed)", "kind": "final", "val_macro_f1": np.nan, "method": f"macro-F1 test {f['macro_f1']['mean']:.4f} ± {f['macro_f1']['std']:.4f}, top-1 {f['top1']['mean']:.4f}"},
    {"exp_id": "T00+I00 (test, 3 seed)", "kind": "baseline", "val_macro_f1": np.nan, "method": f"macro-F1 test {t['macro_f1']['mean']:.4f} ± {t['macro_f1']['std']:.4f}, top-1 {t['top1']['mean']:.4f}"},
    {"exp_id": "Realtime", "kind": "deploy", "val_macro_f1": RT.val_macro_f1, "method": f"{RT.method}: p95 {RT.p95_ms:.1f} ms batch 1"},
])])

LS.write_xlsx("results.xlsx", {"Summary": SUMMARY, "Backbones": btab, "Training": ttab, "Inference": INF,
                               "Final": FINAL, "PerClass": PER, "Latency": LATENCY},
              highlight={"Backbones": "val_macro_f1", "Training": "val_macro_f1", "Inference": "val_macro_f1"})
print("Đã ghi", os.path.abspath("results.xlsx"))
missing = [c for c in pd.concat([btab.curve, ttab.curve]) if not os.path.exists(c)]
print("Ảnh curves thiếu:", missing or "không")

### Việc còn lại (làm tay)
1. Viết `report.md` theo GUIDE mục 6.3 — mọi con số lấy từ `results.xlsx` / `eval_out/`.
2. Tải `results.xlsx`, `curves/`, `predictions/`, `figures/` từ Drive (`MyDrive/DeepWeeds_Lab2`) về `submissions/<mssv>_<ho_ten>/` trong fork, cùng `code/` và README riêng (link notebook này, version thư viện in ở ô đầu).
3. Không commit `images.zip`, `runs/*.pt`.